# Usecase 4: EMP EMPO-3 classification — run ritme models

Launches ritme classification experiments via the shared template `src/run_ritme_model.sh` and the `submit_model` helper in `src/launch_models.py`. Set up the env once (the last command must be run from the repo root):

```shell
mamba create -n ritme_usecases -c adamova -c conda-forge -c bioconda -c pytorch ritme ipykernel nbconvert -y
conda activate ritme_usecases
pip install -e .
```

The splits are pre-staged by `n1_data.ipynb`, which writes EMP's published fixed split directly: the 2,000-sample subset trains, the remaining QC-filtered samples are held out. `submit_model` therefore skips the split step for this use case.

u4 is wider and longer-running than u1-u3, so three settings differ from those notebooks and are set explicitly below: a per-model concurrency cap, a 48 h wall time, and a SHAP background-sample cap.

## Setup

In [ ]:
from src import cluster_config
from src.launch_models import submit_model

## Configuration

In [ ]:
# Where ritme writes experiment outputs, relative to the repo root unless
# absolute. Per-experiment subfolders and sbatch logs land here.
LOGS_DIR = "use_cases/ritme_runs/local"

# Cluster account and any nodes to avoid are site-specific: they come from
# .cluster.json (gitignored) or the RITME_* environment variables.
# See src/cluster_config.py.
SLURM_ACCOUNT = cluster_config.slurm_account()
EXCLUDE_NODES = cluster_config.exclude_nodes()

# Trials run concurrently inside one job. Each holds its own copy of a
# ~3.1e5-feature frame, so the cap is set by memory per trial, not cores.
MAX_CONCURRENT_TRIALS = {"logreg": 8, "rf_class": 8, "xgb_class": 5, "nn_class": 5}

# evaluate-tuned-models, the bootstrap and SHAP all run after the search on
# a 51 GiB test frame; the default one-hour buffer over time_budget_s is too
# thin for that tail.
SLURM_TIME = "48:00:00"

# SHAP scales with samples x features x classes and is intractable here with
# the full training set as background (~3.1e5 features, 15 classes).
SHAP_MAX_BACKGROUND_SAMPLES = "100"

## Launch experiments

One sbatch job per model class, using the `("u4", model)` rows of `src/launch_models.py:SLURM_RESOURCES`.

Excluded by design: `trac` and the regression-only models (`linreg`, `rf`, `xgb`, `nn_reg`), and `nn_corn` (ordinal — EMPO-3 is nominal).

In [ ]:
import os

# Read by `ritme explain-features` through the run template.
os.environ["SHAP_MAX_BACKGROUND_SAMPLES"] = SHAP_MAX_BACKGROUND_SAMPLES

models = ["logreg", "rf_class", "xgb_class", "nn_class"]

for model_type in models:
    submit_model(
        "u4",
        model_type,
        sampler="tpe",
        mode="slurm",
        logs_dir=LOGS_DIR,
        slurm_account=SLURM_ACCOUNT,
        slurm_time=SLURM_TIME,
        max_concurrent_trials=MAX_CONCURRENT_TRIALS[model_type],
        sbatch_extra=(
            [f"--exclude={','.join(EXCLUDE_NODES)}"] if EXCLUDE_NODES else None
        ),
    )

## Smoke test (optional)

Validate the dispatch end-to-end with one short search before committing the full budget. It runs the same pipeline, into a separate logs dir so it cannot collide with a real run.

In [ ]:
# submit_model(
#     "u4", "logreg",
#     mode="slurm", logs_dir="x_scratch/smoke_runs",
#     slurm_account=SLURM_ACCOUNT,
#     max_concurrent_trials=MAX_CONCURRENT_TRIALS["logreg"],
#     config_overrides={"experiment_tag": "smoke_u4_logreg_tpe",
#                       "time_budget_s": 600},
# )